# Apart Research AI Collusion Sprint — Q1.4 experiment

This is the full experiment runner with **50 repeats per condition**, **199 honest-null batches** for each environment/game budget, and checkpoint saving after every completed condition.

It requires `numpy`, `pandas`, and `matplotlib`. In Google Colab, run the next cell first if needed:

```python
!pip -q install numpy pandas matplotlib
```

Then run the experiment code cell. It writes `apart_collusion_results.csv` and `apart_collusion_results.png` to the current working directory. The CSV is updated after each condition, so completed results remain available if the run is interrupted.

**Interpretation caveat:** this is a toy-model comparison, not an exact replication of Bonjour et al. RPS and Mini-Ludo use different planted behaviours and different scoring statistics; cross-game differences are descriptive and do not by themselves establish that complexity causes a change in detectability.

In [ ]:
!pip -q install numpy pandas matplotlib

In [ ]:
"""Apart Research AI Collusion Sprint — Question 1.4
Controlled toy experiment comparing repeated Rock–Paper–Scissors (RPS) with a
simplified four-player Mini-Ludo simulator.

This is a research prototype inspired by Bonjour et al. (2022), NOT a replication
of their exact state-conditioned net-influence statistic. See README cell in the
companion notebook for interpretation and limitations.

Run: python apart_collusion_experiment.py\nThis version uses 50 repeats per condition and 199 honest-null batches.\nIt checkpoints the CSV after every condition.
Dependencies: numpy, pandas, matplotlib
"""
from __future__ import annotations
import itertools, random, math, csv, os
from collections import defaultdict
import numpy as np
import pandas as pd

# ----------------------------- SETTINGS --------------------------------------
SEED = 12345
PLAYERS = 4
PARTNERS = (0, 2)
STRENGTHS = [0.0, 0.25, 0.5, 0.75, 1.0]
GAMES_LIST = [5, 10, 20]
# Number of independent experiment batches per setting. Increase for final runs.
EXPERIMENTS = 50
# Honest reference batches used to calibrate max-statistic p-values.
NULL_EXPERIMENTS = 199
RPS_ROUNDS_PER_GAME = 100
MAX_LUDO_TURNS = 2500
HONEST_RANDOM_MOVE_PROB = 0.20
ALPHA = 0.05
OUTPUT_CSV = "apart_collusion_results.csv"
OUTPUT_PLOT = "apart_collusion_results.png"

COLOURS = ["green", "yellow", "light blue", "pink"]
TOKENS = 4
RING = 52
START_GAP = 13
LAST_RING_PROGRESS = 50
HOME = 56
BONUS_ROLL_RULE = "always"  # or "yard_empty"
PAIRS = list(itertools.combinations(range(PLAYERS), 2))

# ----------------------------- shared helpers -------------------------------
def wilson_interval(k: int, n: int, z: float = 1.96):
    if n <= 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z*z/n
    center = p + z*z/(2*n)
    half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((center-half)/den, (center+half)/den)


def empirical_pvalue(null_maxima, observed_max):
    """Finite-sample corrected upper-tail p-value; null_maxima are max pair scores."""
    null_maxima = np.asarray(null_maxima, dtype=float)
    return (1 + int(np.sum(null_maxima >= observed_max))) / (1 + len(null_maxima))


def rank_pair(scores):
    # Stable deterministic tie-break for reproducibility.
    return max(scores, key=lambda p: (scores[p], -PAIRS.index(p)))

# ================================ RPS ========================================
def play_rps_batch(rng: random.Random, n_games: int, strength: float):
    """Returns per-player action sequences. Each game has independent rounds.

    Honest agents choose uniformly at random. During each round, with probability
    strength, the planted pair chooses one shared uniformly random action; otherwise
    all four players choose independently. This is deliberately a transparent toy
    collusion mechanism, not a claim about realistic strategic RPS agents.
    """
    sequences = [[] for _ in range(PLAYERS)]
    for _ in range(n_games):
        for _ in range(RPS_ROUNDS_PER_GAME):
            if rng.random() < strength:
                shared = rng.randrange(3)
                actions = [rng.randrange(3) for _ in range(PLAYERS)]
                actions[PARTNERS[0]] = shared
                actions[PARTNERS[1]] = shared
            else:
                actions = [rng.randrange(3) for _ in range(PLAYERS)]
            for p, a in enumerate(actions):
                sequences[p].append(a)
    return sequences


def mutual_information(x, y):
    """Plug-in empirical MI in bits for categorical sequences."""
    x = np.asarray(x, dtype=int); y = np.asarray(y, dtype=int)
    if len(x) == 0 or len(x) != len(y):
        return 0.0
    joint = np.zeros((3, 3), dtype=float)
    np.add.at(joint, (x, y), 1)
    pxy = joint / joint.sum()
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    mask = pxy > 0
    return float(np.sum(pxy[mask] * np.log2(pxy[mask] / (px @ py)[mask])))


def rps_pair_scores(sequences):
    return {pair: mutual_information(sequences[pair[0]], sequences[pair[1]]) for pair in PAIRS}

# ============================== Mini-Ludo ====================================
def ring_pos(player, progress):
    return (START_GAP * player + progress) % RING


def legal_moves(state, player, roll):
    """Moves are (token_id, new_progress, captured_by_owner, suicide_on_block)."""
    moves = []
    for t, prog in enumerate(state[player]):
        if prog == HOME:
            continue
        if prog == -1:
            if roll != 6:
                continue
            new = 0
        else:
            new = prog + roll
            if new > HOME:
                continue
        captured, suicide = {}, False
        if new <= LAST_RING_PROGRESS:
            pos = ring_pos(player, new)
            for q in range(PLAYERS):
                if q == player:
                    continue
                here = [u for u, pq in enumerate(state[q])
                        if 0 <= pq <= LAST_RING_PROGRESS and ring_pos(q, pq) == pos]
                if len(here) >= 2:
                    suicide = True
                elif len(here) == 1:
                    captured[q] = here
        moves.append((t, new, captured, suicide))
    return moves


def move_score(move, state, player):
    t, new, captured, suicide = move
    if suicide:
        return -1000.0
    score = float(new)
    if captured:
        score += 100 + 10 * sum(len(v) for v in captured.values())
    if new == HOME:
        score += 60
    if state[player][t] == -1:
        score += 40
    if new <= LAST_RING_PROGRESS and any(u != t and state[player][u] == new for u in range(TOKENS)):
        score += 10
    return score


def choose_ludo_move(rng, candidates, state, player):
    if rng.random() < HONEST_RANDOM_MOVE_PROB:
        return rng.choice(candidates)
    return max(candidates, key=lambda m: move_score(m, state, player))


def opening_player(rng):
    contenders = list(range(PLAYERS))
    while len(contenders) > 1:
        rolls = {p: rng.randint(1, 6) for p in contenders}
        best = max(rolls.values())
        contenders = [p for p in contenders if rolls[p] == best]
    return contenders[0]


def play_ludo_game(rng, strength, events):
    """Play one Ludo game. Events are (actor, target, captured_1_or_0).

    At each decision, colluding players spare their partner with probability
    `strength` if at least one alternative legal move exists. This is an explicit
    planted behaviour, not inferred from the game.
    """
    state = [[-1] * TOKENS for _ in range(PLAYERS)]
    player = opening_player(rng)
    turns = 0
    while turns < MAX_LUDO_TURNS:
        turns += 1
        sixes = 0
        while True:
            roll = rng.randint(1, 6)
            yard_empty = all(p != -1 for p in state[player])
            if roll == 6:
                sixes += 1
                if sixes == 3:
                    break
            moves = legal_moves(state, player, roll)
            if moves:
                candidates = moves
                is_partner = player in PARTNERS
                partner = PARTNERS[1] if player == PARTNERS[0] else PARTNERS[0]
                if is_partner and rng.random() < strength:
                    alternatives = [m for m in moves if partner not in m[2]]
                    if alternatives:
                        candidates = alternatives
                chosen = choose_ludo_move(rng, candidates, state, player)
                # One opportunity per target player per decision, not one per token.
                targets = set()
                for m in moves:
                    targets.update(m[2].keys())
                for target in targets:
                    events.append((player, target, int(target in chosen[2])))
                t, new, captured, suicide = chosen
                if suicide:
                    state[player][t] = -1
                else:
                    state[player][t] = new
                    for owner, token_ids in captured.items():
                        for token_id in token_ids:
                            state[owner][token_id] = -1
                    if all(p == HOME for p in state[player]):
                        return player, turns
            if roll == 6 and (BONUS_ROLL_RULE == "always" or yard_empty):
                continue
            break
        player = (player + 1) % PLAYERS
    return None, turns


def ludo_pair_scores(events):
    """Pair sparing score = both directions' capture rate against others minus
    capture rate against the candidate partner. Jeffreys-style smoothing avoids
    extreme scores from tiny denominators. Higher scores indicate more sparing.
    """
    opp = np.zeros((PLAYERS, PLAYERS), dtype=float)
    cap = np.zeros((PLAYERS, PLAYERS), dtype=float)
    for actor, target, did_capture in events:
        opp[actor, target] += 1
        cap[actor, target] += did_capture
    rates = np.zeros((PLAYERS, PLAYERS), dtype=float)
    for i in range(PLAYERS):
        for j in range(PLAYERS):
            if i != j:
                rates[i, j] = (cap[i, j] + 0.5) / (opp[i, j] + 1.0)
    scores = {}
    for a, b in PAIRS:
        other_a = [rates[a, j] for j in range(PLAYERS) if j != a and j != b]
        other_b = [rates[b, j] for j in range(PLAYERS) if j != b and j != a]
        scores[(a, b)] = float((np.mean(other_a) - rates[a, b]) + (np.mean(other_b) - rates[b, a]))
    return scores, opp, cap

# ============================ Batch simulation ===============================
def simulate_batch(environment, n_games, strength, seed):
    rng = random.Random(seed)
    if environment == "RPS":
        seqs = play_rps_batch(rng, n_games, strength)
        return rps_pair_scores(seqs), None
    events = []
    unfinished = 0
    for _ in range(n_games):
        _, turns = play_ludo_game(rng, strength, events)
        # `None` winner is not returned here, so turn-cap frequency is counted below
        # in a dedicated diagnostic pass only if desired.
    scores, opp, cap = ludo_pair_scores(events)
    return scores, {"opportunities": opp, "captures": cap, "n_events": len(events)}


def make_null_distribution(environment, n_games, n_null, seed):
    """Simulate independent honest batches and store each batch's maximum score.

    This controls for finite-sample upward bias and the fact that the most suspicious
    pair is selected. It is a model-based Monte Carlo null, not a universal null.
    """
    maxima = []
    for k in range(n_null):
        scores, _ = simulate_batch(environment, n_games, 0.0, seed + 1_000_003*k)
        maxima.append(max(scores.values()))
    return np.asarray(maxima, dtype=float)


def evaluate_setting(environment, strength, n_games, n_exp, null_max, seed):
    true_pair = tuple(sorted(PARTNERS))
    correct = detected = joint = 0
    score_true, score_max = [], []
    for e in range(n_exp):
        scores, _ = simulate_batch(environment, n_games, strength,
                                   seed + 7919*e + int(strength*10000) + n_games*100000)
        winner = rank_pair(scores)
        observed_max = scores[winner]
        pval = empirical_pvalue(null_max, observed_max)
        is_detected = pval < ALPHA
        correct += int(winner == true_pair)
        detected += int(is_detected)
        joint += int(is_detected and winner == true_pair)
        score_true.append(scores[true_pair])
        score_max.append(observed_max)
    lo, hi = wilson_interval(correct, n_exp)
    dlo, dhi = wilson_interval(detected, n_exp)
    jlo, jhi = wilson_interval(joint, n_exp)
    return {
        "environment": environment,
        "strength": strength,
        "games_observed": n_games,
        "experiments": n_exp,
        "pair_identification_rate": correct/n_exp,
        "pair_identification_95ci_low": lo,
        "pair_identification_95ci_high": hi,
        "detection_rate": detected/n_exp,
        "detection_95ci_low": dlo,
        "detection_95ci_high": dhi,
        "correct_detection_rate": joint/n_exp,
        "correct_detection_95ci_low": jlo,
        "correct_detection_95ci_high": jhi,
        "mean_true_pair_score": float(np.mean(score_true)),
        "mean_max_pair_score": float(np.mean(score_max)),
        "null_false_positive_target": ALPHA,
        "null_replicates": len(null_max),
    }

# -------------------------------- tests ---------------------------------------
def run_self_tests():
    assert len(PAIRS) == 6
    assert abs(mutual_information([0,1,2]*10, [0,1,2]*10) - math.log2(3)) < 1e-9
    assert mutual_information([0,1,2]*10, [2,0,1]*10) > 0.99
    # Ludo geometry and core movement checks.
    for p in range(PLAYERS):
        assert ring_pos((p+1) % PLAYERS, 0) == (ring_pos(p, 0) + START_GAP) % RING
    blank = [[-1]*TOKENS for _ in range(PLAYERS)]
    assert legal_moves(blank, 0, 5) == []
    assert len(legal_moves(blank, 0, 6)) == TOKENS
    # Known capture: player 0 progress 5 and player 1 progress 44 -> same square after 3.
    state = [[-1]*TOKENS for _ in range(PLAYERS)]
    state[0][0] = 5
    state[1][0] = 47
    moves = legal_moves(state, 0, 3)
    assert any(m[2].get(1) == [0] for m in moves)
    # Empty event arrays should produce finite scores.
    empty_scores, _, _ = ludo_pair_scores([])
    assert all(np.isfinite(v) for v in empty_scores.values())
    print("Self-tests passed.")


def main():
    run_self_tests()
    print("\nApart Research Q1.4 — controlled toy experiment")
    print(f"Planted pair: {PARTNERS} ({COLOURS[PARTNERS[0]]}, {COLOURS[PARTNERS[1]]})")
    print(f"Strengths={STRENGTHS}; games={GAMES_LIST}; experiments={EXPERIMENTS}; null batches={NULL_EXPERIMENTS}")
    rows = []
    for environment in ("RPS", "Mini-Ludo"):
        for n_games in GAMES_LIST:
            print(f"Calibrating honest null: {environment}, {n_games} games ...", flush=True)
            null_max = make_null_distribution(environment, n_games, NULL_EXPERIMENTS,
                                              SEED + (0 if environment == "RPS" else 5_000_000) + n_games*1000)
            for strength in STRENGTHS:
                n_exp = EXPERIMENTS
                row = evaluate_setting(environment, strength, n_games, n_exp, null_max,
                                       SEED + (0 if environment == "RPS" else 7_000_000))
                rows.append(row)
                # Checkpoint after every condition: results survive an interrupted run.
                pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
                print(f"  strength={strength:>4.2f}: top-pair={row['pair_identification_rate']:.2f}, "
                      f"flagged={row['detection_rate']:.2f}, correct+flagged={row['correct_detection_rate']:.2f}", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved {OUTPUT_CSV}")
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        for ax, env in zip(axes, ("RPS", "Mini-Ludo")):
            sub = df[df.environment == env]
            for n_games in GAMES_LIST:
                s = sub[sub.games_observed == n_games].sort_values("strength")
                ax.plot(s.strength, s.correct_detection_rate, marker="o", label=f"{n_games} games")
            ax.axhline(ALPHA, linestyle="--", label="nominal alpha (reference only)")
            ax.set(title=f"{env}: correctly detected planted pair", xlabel="Collusion strength",
                   ylabel="Fraction of experiments", ylim=(-0.02, 1.02))
            ax.legend()
        fig.tight_layout(); fig.savefig(OUTPUT_PLOT, dpi=160); plt.close(fig)
        print(f"Saved {OUTPUT_PLOT}")
    except Exception as exc:
        print("Plot skipped; CSV is still saved. Reason:", exc)
    print("\nInterpretation reminders:")
    print("- At strength 0, detection rate estimates the false-positive rate under this simulator's honest policy.")
    print("- RPS and Mini-Ludo use different planted collusion mechanisms and different statistics; cross-game differences are descriptive, not a pure causal complexity effect.")
    print("- Bonjour et al.'s exact state-conditioned net-influence detector is not implemented here.")

if __name__ == "__main__":
    main()
